# 📊 Post-Training Quantization và Benchmark: Google Gemma-3-4b-pt

## 🎯 Mục tiêu
Notebook này thực hiện **Post-Training Quantization (PTQ)** trên model `google/gemma-3-4b-it`
sử dụng framework **Unsloth**.

### 📋 Các bước thực hiện:
1. ✅ **Cài đặt thư viện**: Unsloth, transformers, datasets, evaluate
2. ✅ **Load model gốc (Base Model)** ở định dạng 16-bit (float16)
3. ✅ **Quantize model** về 3 dạng: **16-bit, 8-bit, 4-bit**
4. ✅ **Đánh giá benchmark** trên dataset `HuggingFaceH4/MATH-500`
5. ✅ **Tính toán metrics**: BLEU, ROUGE-L
6. ✅ **Trực quan hóa kết quả** bằng biểu đồ so sánh

## 🔧 Bước 1: Cài đặt các thư viện cần thiết

### 📦 Giải thích các thư viện:
- **unsloth**: Framework tối ưu hóa để load và quantize LLMs hiệu quả
  - Hỗ trợ quantization 4-bit, 8-bit cực kỳ nhanh
  - Tích hợp sẵn với bitsandbytes cho QLoRA
- **transformers**: Thư viện Hugging Face để làm việc với pre-trained models
- **datasets**: Load dataset MATH-500 từ Hugging Face Hub
- **evaluate**: Framework đánh giá model với các metrics
- **rouge_score**: Tính ROUGE score (Recall-Oriented Understudy for Gisting Evaluation)
- **sacrebleu**: Tính BLEU score (Bilingual Evaluation Understudy)
- **matplotlib, seaborn**: Trực quan hóa kết quả benchmark

In [1]:
# Cài đặt Unsloth từ GitHub (phiên bản mới nhất)
!pip install -q "unsloth @ git+https://github.com/unslothai/unsloth.git"
!pip -q install unsloth_zoo
# Cài đặt các thư viện hỗ trợ
!pip install -q transformers datasets evaluate rouge_score sacrebleu matplotlib seaborn pandas tqdm accelerate bitsandbytes

print("✅ Đã cài đặt các thư viện thành công!")

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 289.6/289.6 kB 7.6 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 22.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 100.5 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 36.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 224.9/224.9 kB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.3/199.3 kB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.7/146.7 kB 12.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.26.0 requires goog

## 📚 Bước 2: Import thư viện và thiết lập môi trường

### 🔍 Giải thích từng import:
- **torch**: PyTorch framework chính
- **FastLanguageModel**: Class từ Unsloth để load model với quantization tự động
- **AutoTokenizer**: Tokenizer để convert text thành tensor
- **load_dataset**: Load dataset từ Hugging Face Hub
- **evaluate**: Load các metrics BLEU và ROUGE
- **pandas**: Xử lý data dạng bảng
- **matplotlib/seaborn**: Vẽ biểu đồ đẹp

In [2]:
import torch
from unsloth import FastLanguageModel
from transformers import AutoTokenizer
from datasets import load_dataset
import evaluate
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import warnings
import time
import gc  # Garbage collector để giải phóng RAM
warnings.filterwarnings('ignore')

# Thiết lập style đẹp cho biểu đồ
sns.set_style("whitegrid")
sns.set_palette("Set2")
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12

# Kiểm tra GPU
print("="*70)
print(" " * 20 + "THIẾT LẬP MÔI TRƯỜNG")
print("="*70)
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("WARNING: Không tìm thấy GPU. Code vẫn chạy được nhưng sẽ rất chậm!")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


2025-12-22 03:33:43.069839: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1766374423.276964      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1766374423.339202      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1766374423.852947      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1766374423.852990      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1766374423.852994      55 computation_placer.cc:177] computation placer alr

🦥 Unsloth Zoo will now patch everything to make training faster!
                    THIẾT LẬP MÔI TRƯỜNG
CUDA Available: True
GPU Device: Tesla T4
GPU Memory: 15.83 GB


In [3]:
# Định nghĩa constants
MODEL_NAME = "google/gemma-3-4b-it"
DATASET_NAME = "HuggingFaceH4/MATH-500"
MAX_SEQ_LENGTH = 2048
NUM_SAMPLES = 100
print(f"\nModel: {MODEL_NAME}")
print(f"Dataset: {DATASET_NAME}")
print(f"Max Sequence Length: {MAX_SEQ_LENGTH}")
print(f"Number of Samples: {NUM_SAMPLES}")


Model: google/gemma-3-4b-it
Dataset: HuggingFaceH4/MATH-500
Max Sequence Length: 2048
Number of Samples: 100


## 📖 Bước 3: Load Dataset MATH-500

### 💡 Giải thích về MATH-500:
- **MATH-500**: Subset của dataset MATH, chứa 500 bài toán học đa dạng
- **Độ khó**: Từ Pre-Algebra đến Calculus và Number Theory
- **Format**:
  - `problem`: Đề bài toán (input)
  - `solution`: Lời giải chi tiết (reference/ground truth)
  - `level`: Mức độ khó (1-5)
  - `type`: Loại toán (Algebra, Geometry, etc.)

### 🎯 Mục đích:
Đánh giá khả năng reasoning và problem-solving của model sau khi quantization

### ⚠️ Lưu ý với Gemma-3-1b-pt:
Model này là **pre-trained** (chưa fine-tune instruction), nên:
- Có thể không follow instruction tốt
- Output có thể không có cấu trúc rõ ràng
- Performance trên MATH-500 thường thấp hơn instruction-tuned models

In [4]:
print("\n" + "="*70)
print(" " * 22 + "LOADING DATASET")
print("="*70)

# Load dataset
dataset = load_dataset(DATASET_NAME, split="test")
print(f"Đã load dataset với {len(dataset)} mẫu")

# Giới hạn số lượng mẫu (để tăng tốc độ benchmark)
dataset = dataset.select(range(min(NUM_SAMPLES, len(dataset))))
print(f"Sử dụng {len(dataset)} mẫu cho benchmark\n")

# Hiển thị ví dụ
print("VÍ DỤ MẪU SỐ 1:")
print("-" * 70)
print(f"Problem:\n{dataset[0]['problem'][:250]}...")
print(f"\nSolution:\n{dataset[0]['solution'][:250]}...")
print("-" * 70)


                      LOADING DATASET


README.md:   0%|          | 0.00/412 [00:00<?, ?B/s]

test.jsonl: 0.00B [00:00, ?B/s]

Generating test split:   0%|          | 0/500 [00:00<?, ? examples/s]

Đã load dataset với 500 mẫu
Sử dụng 100 mẫu cho benchmark

VÍ DỤ MẪU SỐ 1:
----------------------------------------------------------------------
Problem:
Convert the point $(0,3)$ in rectangular coordinates to polar coordinates.  Enter your answer in the form $(r,\theta),$ where $r > 0$ and $0 \le \theta < 2 \pi.$...

Solution:
We have that $r = \sqrt{0^2 + 3^2} = 3.$  Also, if we draw the line connecting the origin and $(0,3),$ this line makes an angle of $\frac{\pi}{2}$ with the positive $x$-axis.

[asy]
unitsize(0.8 cm);

draw((-0.5,0)--(3.5,0));
draw((0,-0.5)--(0,3.5));...
----------------------------------------------------------------------


## 🔬 Bước 4: Định nghĩa hàm đánh giá (Evaluation Function)

### 📊 Metrics được sử dụng:

#### 1. BLEU (Bilingual Evaluation Understudy)
- **Phạm vi**: 0-100 (càng cao càng tốt)
- **Công dụng**: Đo lường độ chính xác dựa trên n-gram overlap
- **Cách tính**: So sánh n-grams (1-4 words) giữa prediction và reference
- **Ưu điểm**: Đánh giá chính xác cho text generation tasks
- **Nhược điểm**: Không xét đến ngữ nghĩa, chỉ xét exact match

#### 2. ROUGE-L (Recall-Oriented Understudy for Gisting Evaluation - Longest Common Subsequence)
- **Phạm vi**: 0-1 (càng cao càng tốt)
- **Công dụng**: Đo lường longest common subsequence  
- **Cách tính**: Tìm chuỗi con chung dài nhất giữa prediction và reference
- **Ưu điểm**: Tốt cho summarization, không yêu cầu consecutive words
- **Nhược điểm**: Có thể bỏ sót một số chi tiết quan trọng

### ⏱️ Thời gian inference:
Hàm cũng đo thời gian inference để so sánh tốc độ giữa các phiên bản quantized

### 🎯 Lưu ý về prompt engineering cho Gemma:
Gemma-3-1b-pt không được train với instruction format, nên ta sẽ dùng simple completion prompt

In [5]:
# Load metrics
bleu_metric = evaluate.load("sacrebleu")
rouge_metric = evaluate.load("rouge")

def evaluate_model(model, tokenizer, dataset, model_name_display):
    """
    Đánh giá model trên dataset MATH-500 với BLEU và ROUGE metrics

    Args:
        model: Model cần đánh giá (16-bit, 8-bit, hoặc 4-bit)
        tokenizer: Tokenizer tương ứng với model
        dataset: Dataset MATH-500 đã được load
        model_name_display: Tên hiển thị (để debug và báo cáo)

    Returns:
        dict: {
            "BLEU": float (0-100),
            "ROUGE-L": float (0-1),
            "Inference_Time": float (seconds)
        }
    """
    print(f"\nĐang đánh giá model {model_name_display}...")

    predictions = []
    references = []

    # Bắt đầu đo thời gian
    start_time = time.time()

    # Duyệt qua từng mẫu trong dataset
    for idx, item in enumerate(tqdm(dataset, desc=f"Evaluating {model_name_display}")):
        # Tạo prompt đơn giản cho pre-trained model (completion style)
        # Không sử dụng instruction format vì model chưa được fine-tune
        prompt = f"""Question: {item['problem']}

Answer:"""

        # Tokenize input
        inputs = tokenizer(
            text=prompt,
            return_tensors="pt",
            truncation=True,
            max_length=MAX_SEQ_LENGTH
        ).to(model.device)

        # Generate prediction
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=512,  # Giới hạn độ dài output
                temperature=0.7,
                do_sample=True,
                top_p=0.9,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )

        # Decode prediction
        generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

        # Extract chỉ phần answer (bỏ prompt)
        answer = generated_text[len(prompt):].strip()

        predictions.append(answer)
        references.append(item['solution'])

    # Kết thúc đo thời gian
    end_time = time.time()
    inference_time = end_time - start_time

    # Tính BLEU score
    # BLEU yêu cầu references phải là list of lists
    bleu_result = bleu_metric.compute(
        predictions=predictions,
        references=[[ref] for ref in references]
    )

    # Tính ROUGE score
    rouge_result = rouge_metric.compute(
        predictions=predictions,
        references=references
    )

    # Tổng hợp kết quả
    results = {
        "BLEU": bleu_result['score'],
        "ROUGE-L": rouge_result['rougeL'],
        "Inference_Time": inference_time,
        "Avg_Time_Per_Sample": inference_time / len(dataset)
    }

    print(f"Hoàn thành đánh giá {model_name_display}!")
    print(f"BLEU Score: {results['BLEU']:.2f}")
    print(f"ROUGE-L Score: {results['ROUGE-L']:.4f}")
    print(f"Total Time: {results['Inference_Time']:.2f}s")
    print(f"Avg Time/Sample: {results['Avg_Time_Per_Sample']:.2f}s")

    return results

## 🔵 Bước 5: Load và đánh giá Base Model (16-bit Float16)

### 🎯 Mục đích:
Model 16-bit (float16/half precision) sẽ là **baseline** để so sánh với các phiên bản quantized.

### 💡 Giải thích các parameters:
- **model_name**: Tên model trên Hugging Face Hub (google/gemma-3-1b-pt)
- **max_seq_length**: Độ dài sequence tối đa model có thể xử lý (2048 tokens)
- **dtype=torch.float16**: Sử dụng half precision (16-bit) thay vì float32
  - Giảm 50% VRAM so với float32
  - Tốc độ inference nhanh hơn ~2x trên GPU hiện đại
  - Độ chính xác gần như không đổi so với float32
- **load_in_4bit=False**: Không quantize, giữ nguyên float16

### 📈 Kỳ vọng:
Model 16-bit thường đạt performance cao nhất nhưng tốn nhiều VRAM nhất

### ℹ️ Về Gemma-3-1b-pt:
- **Architecture**: GemmaForCausalLM với 1B parameters
- **Training**: Pre-trained on large corpus (chưa instruction tuning)
- **Context Length**: 8K tokens maximum

In [ ]:
print("\n" + "="*70)
print(" " * 18 + "BASE MODEL (16-BIT FLOAT16)")
print("="*70)


# Load model ở dạng float16 (16-bit) - không quantization
model_16bit, tokenizer_16bit = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=torch.float16,  
    load_in_4bit=False,   
)

# Chuyển sang inference mode (tắt gradient computation để nhanh hơn)
model_16bit = FastLanguageModel.for_inference(model_16bit)

print(f"Model Architecture: {model_16bit.__class__.__name__}")
print(f"Model Parameters: {sum(p.numel() for p in model_16bit.parameters()) / 1e9:.2f}B")
print(f"Approx VRAM: ~{sum(p.numel() * 2 for p in model_16bit.parameters()) / 1e9:.2f} GB (FP16)")

# Đánh giá model 16-bit
results_16bit = evaluate_model(model_16bit, tokenizer_16bit, dataset, "16-bit (FP16)")

# Giải phóng bộ nhớ
del model_16bit
gc.collect()
torch.cuda.empty_cache()
print("\nĐã giải phóng bộ nhớ model 16-bit")

## 🟢 Bước 6: Load và đánh giá Model 8-bit (INT8 Quantization)

### 🎯 Mục đích:
Giảm kích thước model xuống ~50% so với 16-bit bằng 8-bit quantization

### 💡 Giải thích kỹ thuật 8-bit quantization:
- **Algorithm**: LLM.int8() từ bitsandbytes
- **Cơ chế**:
  - Weights được quantize từ float16 → int8
  - Activations vẫn giữ nguyên float16
  - Sử dụng mixed-precision matrix multiplication
- **Outlier handling**: Giữ lại một số outliers quan trọng ở float16
- **VRAM saving**: Giảm ~50% so với float16
- **Performance impact**: Thường giảm < 1-2% accuracy
- **Speed**: Tương đương hoặc nhanh hơn float16 một chút

### �� So sánh với 16-bit:
- ✅ Ít VRAM hơn 2x
- ✅ Tốc độ tương đương
- ⚠️ Accuracy giảm nhẹ (~1-2%)

### 🔍 Quan sát với Gemma-3-1b-pt:
Do đây là small model (1B params), quantization impact có thể lớn hơn một chút so với larger models

## 🟡 Bước 7: Load và đánh giá Model 4-bit (NF4 Quantization)

### 🎯 Mục đích:
Giảm kích thước model xuống ~25% so với 16-bit bằng 4-bit quantization

### 💡 Giải thích kỹ thuật 4-bit quantization (QLoRA):
- **Algorithm**: QLoRA (Quantized Low-Rank Adaptation)
- **Data type**: NF4 (Normal Float 4) - phân phối chuẩn 4-bit
- **Cơ chế**:
  - Weights được quantize từ float16 → nf4 (4-bit)
  - Computation vẫn thực hiện ở bfloat16/float16
  - Sử dụng double quantization cho scale factors
- **Nested quantization**: Quantize cả scale factors để tiết kiệm thêm VRAM
- **VRAM saving**: Giảm ~75% so với float16
- **Performance impact**: Thường giảm 3-5% accuracy
- **Speed**: Chậm hơn 16-bit/8-bit một chút do overhead của dequant

### 📊 So sánh với 16-bit:
- ✅ Ít VRAM hơn 4x
- ⚠️ Tốc độ inference chậm hơn ~10-20%
- ⚠️ Accuracy giảm ~3-5%
- ✅ Phù hợp cho fine-tuning trên GPU consumer (RTX 3090, 4090)

### 🎓 Lưu ý:
4-bit quantization rất tốt cho **fine-tuning**, nhưng cho **inference** thì 8-bit thường
là lựa chọn tối ưu hơn (trade-off giữa VRAM và accuracy)

### ⚠️ Quan sát với small models (1B):
Small models như Gemma-3-1b-pt có thể bị degradation cao hơn khi quantize về 4-bit

In [ ]:
print("\n" + "="*70)
print(" " * 17 + "4-BIT MODEL (NF4 QUANTIZATION)")
print("="*70)

# Load model với 4-bit quantization (NF4)
model_4bit, tokenizer_4bit = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,  # Kích hoạt 4-bit quantization (NF4)
)

# Chuyển sang inference mode
model_4bit=FastLanguageModel.for_inference(model_4bit)

print(f"Đã load model 4-bit thành công!")
print(f"Approx VRAM: ~{sum(p.numel() * 0.5 for p in model_4bit.parameters()) / 1e9:.2f} GB (NF4)")
print(f"VRAM reduction: ~75% so với 16-bit")

# Đánh giá model 4-bit
results_4bit = evaluate_model(model_4bit, tokenizer_4bit, dataset, "4-bit (NF4)")

# Giải phóng bộ nhớ
del model_4bit
gc.collect()
torch.cuda.empty_cache()
print("\nĐã giải phóng bộ nhớ model 4-bit")

In [ ]:
print("\n" + "="*70)
print(" " * 18 + "8-BIT MODEL (INT8 QUANTIZATION)")
print("="*70)

# Load model với 8-bit quantization
model_8bit, tokenizer_8bit = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,  # Auto-detect
    load_in_4bit=False,
    load_in_8bit=True,  # Kích hoạt 8-bit quantization
)

# Chuyển sang inference mode
model_8bit=FastLanguageModel.for_inference(model_8bit)

print(f"Đã load model 8-bit thành công!")
print(f"Approx VRAM: ~{sum(p.numel() for p in model_8bit.parameters()) / 1e9:.2f} GB (INT8)")
print(f"VRAM reduction: ~50% so với 16-bit")

# Đánh giá model 8-bit
results_8bit = evaluate_model(model_8bit, tokenizer_8bit, dataset, "8-bit (INT8)")

# Giải phóng bộ nhớ
del model_8bit
gc.collect()
torch.cuda.empty_cache()
print("\nĐã giải phóng bộ nhớ model 8-bit")

## 📊 Bước 8: Tổng hợp và trực quan hóa kết quả

### 📈 Biểu đồ sẽ bao gồm:
1. **Bar chart**: So sánh BLEU và ROUGE-L scores giữa 3 phiên bản
2. **Line plot**: Thời gian inference trung bình mỗi sample
3. **Combined comparison**: Overview tổng thể

### 🎯 Mục đích:
Giúp dễ dàng nhận biết trade-off giữa:
- **Accuracy** (BLEU, ROUGE-L)
- **Efficiency** (VRAM, inference time)

In [ ]:
print("\n" + "="*70)
print(" " * 20 + "TỔNG HỢP KẾT QUẢ")
print("="*70)

# Tạo DataFrame để dễ visualize
results_df = pd.DataFrame({
    'Model': ['16-bit (FP16)', '8-bit (INT8)', '4-bit (NF4)'],
    'BLEU': [results_16bit['BLEU'], results_8bit['BLEU'], results_4bit['BLEU']],
    'ROUGE-L': [results_16bit['ROUGE-L'], results_8bit['ROUGE-L'], results_4bit['ROUGE-L']],
    'Inference_Time': [results_16bit['Inference_Time'], results_8bit['Inference_Time'], results_4bit['Inference_Time']],
    'Avg_Time_Per_Sample': [results_16bit['Avg_Time_Per_Sample'], results_8bit['Avg_Time_Per_Sample'], results_4bit['Avg_Time_Per_Sample']],
    'VRAM_Reduction': ['Baseline (100%)', '-50%', '-75%']
})

print("\nBẢNG KẾT QUẢ CHI TIẾT:")
print(results_df.to_string(index=False))

# Tính % degradation so với baseline
print("\nPERFORMANCE DEGRADATION SO VỚI BASELINE (16-bit):")
print(f"  8-bit BLEU: {((results_8bit['BLEU'] - results_16bit['BLEU']) / max(results_16bit['BLEU'], 0.01) * 100):+.2f}%")
print(f"  8-bit ROUGE-L: {((results_8bit['ROUGE-L'] - results_16bit['ROUGE-L']) / max(results_16bit['ROUGE-L'], 0.01) * 100):+.2f}%")
print(f"  4-bit BLEU: {((results_4bit['BLEU'] - results_16bit['BLEU']) / max(results_16bit['BLEU'], 0.01) * 100):+.2f}%")
print(f"  4-bit ROUGE-L: {((results_4bit['ROUGE-L'] - results_16bit['ROUGE-L']) / max(results_16bit['ROUGE-L'], 0.01) * 100):+.2f}%")

## 📈 Bước 9: Vẽ biểu đồ so sánh (Visualization)

### 🎨 Biểu đồ 1: So sánh BLEU và ROUGE-L Scores
Biểu đồ cột (bar chart) so sánh trực tiếp performance metrics

### ⏱️ Biểu đồ 2: So sánh thời gian Inference
Biểu đồ cột so sánh tốc độ inference

### 🔄 Biểu đồ 3: Trade-off Analysis
Line plot kết hợp để thấy được trade-off giữa accuracy và efficiency

In [ ]:
# Tạo figure với 3 subplots
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle(f'Benchmark: Post-Training Quantization - {MODEL_NAME}\nDataset: {DATASET_NAME} ({NUM_SAMPLES} samples)',
             fontsize=16, fontweight='bold')

# ============= SUBPLOT 1: BLEU Score Comparison =============
ax1 = axes[0, 0]
bars1 = ax1.bar(results_df['Model'], results_df['BLEU'],
                color=['#3498db', '#2ecc71', '#f39c12'], alpha=0.8, edgecolor='black')
ax1.set_ylabel('BLEU Score', fontweight='bold')
ax1.set_title('BLEU Score Comparison\n(Higher is better)', fontweight='bold')
ax1.grid(axis='y', alpha=0.3)

# Thêm giá trị lên đầu mỗi cột
for bar in bars1:
    height = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2., height,
             f'{height:.2f}',
             ha='center', va='bottom', fontweight='bold')

# ============= SUBPLOT 2: ROUGE-L Score Comparison =============
ax2 = axes[0, 1]
bars2 = ax2.bar(results_df['Model'], results_df['ROUGE-L'],
                color=['#3498db', '#2ecc71', '#f39c12'], alpha=0.8, edgecolor='black')
ax2.set_ylabel('ROUGE-L F1 Score', fontweight='bold')
ax2.set_title('ROUGE-L Score Comparison\n(Higher is better)', fontweight='bold')
ax2.grid(axis='y', alpha=0.3)

for bar in bars2:
    height = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2., height,
             f'{height:.4f}',
             ha='center', va='bottom', fontweight='bold')

# ============= SUBPLOT 3: Inference Time Comparison =============
ax3 = axes[1, 0]
bars3 = ax3.bar(results_df['Model'], results_df['Avg_Time_Per_Sample'],
                color=['#e74c3c', '#9b59b6', '#1abc9c'], alpha=0.8, edgecolor='black')
ax3.set_ylabel('Seconds per Sample', fontweight='bold')
ax3.set_title('Average Inference Time\n(Lower is better)', fontweight='bold')
ax3.grid(axis='y', alpha=0.3)

for bar in bars3:
    height = bar.get_height()
    ax3.text(bar.get_x() + bar.get_width()/2., height,
             f'{height:.2f}s',
             ha='center', va='bottom', fontweight='bold')

# ============= SUBPLOT 4: Combined Normalized Comparison =============
ax4 = axes[1, 1]

# Normalize scores to 0-100 scale for easy comparison
bleu_max = max(results_df['BLEU'].max(), 0.01)  # Avoid division by zero
rouge_max = max(results_df['ROUGE-L'].max(), 0.01)
time_max = max(results_df['Avg_Time_Per_Sample'].max(), 0.01)

bleu_norm = (results_df['BLEU'] / bleu_max) * 100
rouge_norm = (results_df['ROUGE-L'] / rouge_max) * 100
# Normalize inference time (inverse - faster is better)
time_norm = (1 - (results_df['Avg_Time_Per_Sample'] / time_max)) * 100

x = range(len(results_df['Model']))
width = 0.25

bars_bleu = ax4.bar([i - width for i in x], bleu_norm, width, label='BLEU (normalized)',
                     color='#3498db', alpha=0.8, edgecolor='black')
bars_rouge = ax4.bar([i for i in x], rouge_norm, width, label='ROUGE-L (normalized)',
                      color='#2ecc71', alpha=0.8, edgecolor='black')
bars_speed = ax4.bar([i + width for i in x], time_norm, width, label='Speed Score (inverse time)',
                      color='#e74c3c', alpha=0.8, edgecolor='black')

ax4.set_ylabel('Normalized Score (%)', fontweight='bold')
ax4.set_title('Combined Performance Comparison\n(Normalized to 0-100)', fontweight='bold')
ax4.set_xticks(x)
ax4.set_xticklabels(results_df['Model'])
ax4.legend(loc='lower right')
ax4.grid(axis='y', alpha=0.3)
ax4.set_ylim([0, 105])

plt.tight_layout()
plt.savefig('/kaggle/working//Gemma-3-4b-benchmark-results.png',
            dpi=150, bbox_inches='tight')
print("\nĐã lưu biểu đồ tại: /Gemma-3-1b-benchmark-results.png")
plt.show()

In [6]:
!pip install -q openai

In [7]:
import torch
import numpy as np
from tqdm import tqdm
import json
from typing import Dict, List, Tuple
from openai import OpenAI
import time

In [8]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
openai_api_key = user_secrets.get_secret("api_key")

In [9]:
client = OpenAI(api_key=openai_api_key)

# Xây dựng các hàm

In [10]:
# ================================================================================
# PHẦN 1: CHUẨN BỊ DATASET
# ================================================================================

def prepare_dataset(dataset, num_samples=25):
    """
    Chuẩn bị dataset và tạo references

    Args:
        dataset: Dataset từ HuggingFace
        num_samples: Số lượng samples để đánh giá

    Returns:
        tuple: (dataset_subset, references)
    """
    print(f"\nChuẩn bị {num_samples} samples từ dataset...")

    # Lấy subset
    dataset_subset = dataset.select(range(min(num_samples, len(dataset))))

    # Tạo references (ground truth)
    references = [item['solution'] for item in dataset_subset]

    print(f"Đã chuẩn bị {len(dataset_subset)} samples")
    print(f"Đã tạo {len(references)} references\n")

    return dataset_subset, references


In [12]:
# ================================================================================
# PHẦN 2: HÀM INFERENCE CHO TỪNG MODEL
# ================================================================================

def inference_single_question(model, tokenizer, question, max_new_tokens=512, temperature=0.7):
    """
    Inference cho 1 câu hỏi

    Args:
        model: Model đã load
        tokenizer: Tokenizer tương ứng
        question: Câu hỏi (problem)
        max_new_tokens: Số token tối đa để generate
        temperature: Temperature cho sampling

    Returns:
        str: Câu trả lời của model
    """
    # Tạo prompt
    prompt = f"""Question: {question}

Answer:"""

    # Tokenize
    inputs = tokenizer(
        text=prompt,
        return_tensors="pt",
        truncation=True,
        max_length=2048
    ).to(model.device)

    # Generate
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=True,
            top_p=0.9,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    # Decode
    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Extract answer (bỏ prompt)
    answer = generated_text[len(prompt):].strip()

    return answer


def batch_inference(model, tokenizer, dataset_subset, model_name="Model"):
    """
    Inference cho toàn bộ dataset

    Args:
        model: Model đã load
        tokenizer: Tokenizer tương ứng
        dataset_subset: Dataset đã chuẩn bị
        model_name: Tên model (để hiển thị)

    Returns:
        list: Danh sách predictions
    """
    print(f"\nĐang inference với {model_name}...")

    predictions = []

    for idx, item in enumerate(tqdm(dataset_subset, desc=f"Inference {model_name}")):
        question = item['problem']
        answer = inference_single_question(model, tokenizer, question)
        predictions.append(answer)

    print(f"Hoàn thành inference cho {model_name}: {len(predictions)} predictions\n")

    return predictions


In [13]:
# ================================================================================
# PHẦN 3: ĐÁNH GIÁ VỚI METRICS
# ================================================================================

def calculate_metrics(predictions, references, bleu_metric, rouge_metric):
    """
    Tính toán BLEU và ROUGE scores

    Args:
        predictions: Danh sách predictions
        references: Danh sách references (ground truth)
        bleu_metric: BLEU metric object
        rouge_metric: ROUGE metric object

    Returns:
        dict: {"bleu": float, "rouge_l": float}
    """
    # Tính BLEU
    bleu_result = bleu_metric.compute(
        predictions=predictions,
        references=[[ref] for ref in references]
    )

    # Tính ROUGE
    rouge_result = rouge_metric.compute(
        predictions=predictions,
        references=references
    )

    return {
        "bleu": bleu_result['score'],
        "rouge_l": rouge_result['rougeL']
    }


In [26]:
# ================================================================================
# PHẦN 4: TỔNG HỢP KẾT QUẢ
# ================================================================================

def create_results_dict(predictions_16bit, predictions_8bit,
                       references, bleu_metric, rouge_metric):
    """
    Tạo dictionary kết quả cho tất cả models

    Args:
        predictions_16bit: Predictions từ model 16-bit
        predictions_8bit: Predictions từ model 8-bit
        references: Ground truth references
        bleu_metric: BLEU metric object
        rouge_metric: ROUGE metric object

    Returns:
        dict: Kết quả đầy đủ cho tất cả models
    """
    print("\nĐang tính toán metrics cho tất cả models...\n")

    results = {}

    # Model 16-bit
    print("Tính metrics cho 16-bit model...")
    metrics_16bit = calculate_metrics(predictions_16bit, references, bleu_metric, rouge_metric)
    results["16-bit"] = {
        "predictions": predictions_16bit,
        "bleu": metrics_16bit["bleu"],
        "rouge_l": metrics_16bit["rouge_l"]
    }
    print(f"   BLEU: {metrics_16bit['bleu']:.2f}, ROUGE-L: {metrics_16bit['rouge_l']:.4f}")

    # Model 8-bit
    print("Tính metrics cho 8-bit model...")
    metrics_8bit = calculate_metrics(predictions_8bit, references, bleu_metric, rouge_metric)
    results["8-bit"] = {
        "predictions": predictions_8bit,
        "bleu": metrics_8bit["bleu"],
        "rouge_l": metrics_8bit["rouge_l"]
    }
    print(f"   BLEU: {metrics_8bit['bleu']:.2f}, ROUGE-L: {metrics_8bit['rouge_l']:.4f}")    

    print("\nHoàn thành tính toán metrics!\n")

    return results


In [28]:
# ================================================================================
# PHẦN 5: LLM-AS-A-JUDGE EVALUATION
# ================================================================================

def evaluate_all_models_with_gpt(problem, predictions_dict, reference, openai_client):
    """
    Sử dụng GPT để đánh giá chất lượng câu trả lời của TẤT CẢ các models cùng lúc

    Args:
        problem: Đề bài toán
        predictions_dict: Dictionary chứa predictions của tất cả models
                         {"16-bit": "answer1", "8-bit": "answer2", "4-bit": "answer3"}
        reference: Lời giải chuẩn
        openai_client: OpenAI client object

    Returns:
        dict: {
            "16-bit": {"score": int, "reasoning": str},
            "8-bit": {"score": int, "reasoning": str},
            "4-bit": {"score": int, "reasoning": str}
        }
    """

    prompt = f"""You are an expert mathematics teacher evaluating student solutions from 3 different AI models.

Problem:
{problem}

Reference Solution (Ground Truth):
{reference}

Model Solutions to Evaluate:

1. Model 16-bit (Base Model):
{predictions_dict.get('16-bit', 'N/A')}

2. Model 8-bit (Quantized):
{predictions_dict.get('8-bit', 'N/A')}


Task: Evaluate EACH model solution on a scale of 1-10, where:
- 1-3: Completely wrong or irrelevant
- 4-5: Partially correct with major errors
- 6-7: Mostly correct with minor errors
- 8-9: Correct with small imperfections
- 10: Perfect solution

Provide your evaluation in JSON format with the following structure:
{{
    "16-bit": {{
        "score": <number 1-10>,
        "reasoning": "<brief explanation in 1-2 sentences>"
    }},
    "8-bit": {{
        "score": <number 1-10>,
        "reasoning": "<brief explanation in 1-2 sentences>"
    }}
}}

Only return valid JSON, nothing else.
"""

    try:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": "You are a mathematics evaluation expert. Always respond with valid JSON only."},
                {"role": "user", "content": prompt}
            ],
            temperature=0.3
        )

        result_text = response.choices[0].message.content.strip()
        result = json.loads(result_text)

        return result

    except Exception as e:
        print(f"Error with GPT API: {e}")
        return {
            "16-bit": {"score": 0, "reasoning": f"API Error: {str(e)}"},
            "8-bit": {"score": 0, "reasoning": f"API Error: {str(e)}"}
        }


def run_llm_judge_evaluation(dataset_subset, results, references, openai_client, max_samples=25):
    """
    Chạy đánh giá LLM-as-a-Judge cho tất cả models

    Args:
        dataset_subset: Dataset đã chuẩn bị
        results: Dictionary kết quả từ create_results_dict()
        references: Ground truth references
        openai_client: OpenAI client object
        max_samples: Số lượng samples tối đa để đánh giá

    Returns:
        dict: Kết quả đánh giá LLM Judge cho tất cả models
    """
    print("\n" + "="*70)
    print(" " * 20 + "LM-AS-A-JUDGE EVALUATION")
    print("="*70)

    llm_judge_results = {
        "16-bit": {"scores": [], "reasonings": []},
        "8-bit": {"scores": [], "reasonings": []}
    }

    num_samples = min(max_samples, len(dataset_subset))
    print(f"\nĐánh giá {num_samples} samples...\n")

    for idx in tqdm(range(num_samples), desc="Evaluating all models"):
        problem = dataset_subset[idx]["problem"]
        reference = references[idx]

        predictions_dict = {
            "16-bit": results["16-bit"]["predictions"][idx],
            "8-bit": results["8-bit"]["predictions"][idx],
        }

        evaluation = evaluate_all_models_with_gpt(problem, predictions_dict, reference, openai_client)

        for model_name in ["16-bit", "8-bit"]:
            if model_name in evaluation:
                llm_judge_results[model_name]["scores"].append(evaluation[model_name]["score"])
                llm_judge_results[model_name]["reasonings"].append(evaluation[model_name]["reasoning"])

    # Tính điểm trung bình
    for model_name in ["16-bit", "8-bit"]:
        scores = llm_judge_results[model_name]["scores"]
        llm_judge_results[model_name]["avg_score"] = np.mean(scores) if scores else 0
        print(f"\n{model_name} - Average LLM Judge Score: {llm_judge_results[model_name]['avg_score']:.2f}/10")

    print("\nHoàn thành đánh giá LLM-as-a-Judge!\n")

    return llm_judge_results


In [29]:
# ================================================================================
# PHẦN 6: HELPER FUNCTIONS
# ================================================================================

def print_sample_comparison(idx, dataset_subset, results, references):
    """
    In ra so sánh chi tiết cho 1 sample

    Args:
        idx: Index của sample
        dataset_subset: Dataset
        results: Kết quả từ create_results_dict()
        references: Ground truth
    """
    print("\n" + "="*70)
    print(f"SAMPLE {idx + 1} COMPARISON")
    print("="*70)

    print(f"\nProblem:\n{dataset_subset[idx]['problem'][:200]}...\n")

    print(f"Ground Truth:\n{references[idx][:200]}...\n")

    print(f"16-bit Answer:\n{results['16-bit']['predictions'][idx][:200]}...\n")

    print(f"8-bit Answer:\n{results['8-bit']['predictions'][idx][:200]}...\n")

    print("="*70)

# dataset = load_dataset("HuggingFaceH4/MATH-500", split="test")

# Chuẩn bị dataset và references

In [14]:
NUM_SAMPLES = 25  # Số lượng samples để đánh giá
dataset_subset, references = prepare_dataset(dataset, num_samples=NUM_SAMPLES)

print(f"Dataset subset: {len(dataset_subset)} samples")
print(f"References: {len(references)} items")


Chuẩn bị 25 samples từ dataset...
Đã chuẩn bị 25 samples
Đã tạo 25 references

Dataset subset: 25 samples
References: 25 items


# model_16bit, tokenizer_16bit = FastLanguageModel.from_pretrained(...)
# model_8bit, tokenizer_8bit = FastLanguageModel.from_pretrained(..., load_in_8bit=True)
# model_4bit, tokenizer_4bit = FastLanguageModel.from_pretrained(..., load_in_4bit=True)

In [31]:
model_16bit, tokenizer_16bit = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=torch.float16,  # Half precision
    load_in_4bit=False,   # Không quantize
)

# Chuyển sang inference mode (tắt gradient computation để nhanh hơn)
model_16bit = FastLanguageModel.for_inference(model_16bit)

==((====))==  Unsloth 2025.12.8: Fast Gemma3 patching. Transformers: 4.57.1.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.4.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for gemma3 won't work! Using float32.
Unsloth: Gemma3 does not support SDPA - switching to fast eager.
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.96G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/3.64G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/4.69M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/670 [00:00<?, ?B/s]

In [32]:
# Load model với 8-bit quantization
model_8bit, tokenizer_8bit = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,  # Auto-detect
    load_in_4bit=False,
    load_in_8bit=True,  # Kích hoạt 8-bit quantization
)

# Chuyển sang inference mode
model_8bit=FastLanguageModel.for_inference(model_8bit)

==((====))==  Unsloth 2025.12.8: Fast Gemma3 patching. Transformers: 4.57.1.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.4.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for gemma3 won't work! Using float32.
Unsloth: Gemma3 does not support SDPA - switching to fast eager.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [33]:
# Inference với model 16-bit
predictions_16bit = batch_inference(
    model=model_16bit,
    tokenizer=tokenizer_16bit,
    dataset_subset=dataset_subset,
    model_name="16-bit (FP16)"
)

# Inference với model 8-bit
predictions_8bit = batch_inference(
    model=model_8bit,
    tokenizer=tokenizer_8bit,
    dataset_subset=dataset_subset,
    model_name="8-bit (INT8)"
)

# # Inference với model 4-bit
# predictions_4bit = batch_inference(
#     model=model_4bit,
#     tokenizer=tokenizer_4bit,
#     dataset_subset=dataset_subset,
#     model_name="4-bit (NF4)"
# )


Đang inference với 16-bit (FP16)...


Inference 16-bit (FP16): 100%|██████████| 25/25 [14:49<00:00, 35.57s/it]


Hoàn thành inference cho 16-bit (FP16): 25 predictions


Đang inference với 8-bit (INT8)...


Inference 8-bit (INT8): 100%|██████████| 25/25 [40:27<00:00, 97.12s/it] 

Hoàn thành inference cho 8-bit (INT8): 25 predictions



# TÍNH TOÁN METRICS (BLEU, ROUGE)

In [34]:
import evaluate
bleu_metric = evaluate.load("sacrebleu")
rouge_metric = evaluate.load("rouge")

# Tạo results dictionary với tất cả metrics
results = create_results_dict(
    predictions_16bit=predictions_16bit,
    predictions_8bit=predictions_8bit,
    # predictions_4bit=predictions_4bit,
    references=references,
    bleu_metric=bleu_metric,
    rouge_metric=rouge_metric
)

# In kết quả
print("\n" + "="*70)
print("TỔNG HỢP KẾT QUẢ METRICS")
print("="*70)
for model_name in ["16-bit", "8-bit"]:
    print(f"\n{model_name}:")
    print(f"   BLEU: {results[model_name]['bleu']:.2f}")
    print(f"   ROUGE-L: {results[model_name]['rouge_l']:.4f}")


Đang tính toán metrics cho tất cả models...

Tính metrics cho 16-bit model...
   BLEU: 18.12, ROUGE-L: 0.2814
Tính metrics cho 8-bit model...
   BLEU: 18.34, ROUGE-L: 0.2813

Hoàn thành tính toán metrics!


TỔNG HỢP KẾT QUẢ METRICS

16-bit:
   BLEU: 18.12
   ROUGE-L: 0.2814

8-bit:
   BLEU: 18.34
   ROUGE-L: 0.2813


# ĐÁNH GIÁ VỚI LLM-AS-A-JUDGE

In [35]:
llm_judge_results = run_llm_judge_evaluation(
    dataset_subset=dataset_subset,
    results=results,
    references=references,
    openai_client=client,
    max_samples=10  
)

# In kết quả LLM Judge
print("\n" + "="*70)
print("KẾT QUẢ LLM-AS-A-JUDGE")
print("="*70)
for model_name in ["16-bit", "8-bit"]:
    print(f"\n{model_name}:")
    print(f"   Average Score: {llm_judge_results[model_name]['avg_score']:.2f}/10")
    print(f"   Sample Reasonings:")
    for i, reasoning in enumerate(llm_judge_results[model_name]['reasonings'][:3]):
        print(f"      {i+1}. {reasoning}")



                    LM-AS-A-JUDGE EVALUATION

Đánh giá 10 samples...



Evaluating all models: 100%|██████████| 10/10 [00:29<00:00,  2.98s/it]


16-bit - Average LLM Judge Score: 7.30/10

8-bit - Average LLM Judge Score: 8.50/10

Hoàn thành đánh giá LLM-as-a-Judge!


KẾT QUẢ LLM-AS-A-JUDGE

16-bit:
   Average Score: 7.30/10
   Sample Reasonings:
      1. The solution correctly applies the formulas for converting rectangular to polar coordinates, accurately identifies the angle, and provides the correct final answer.
      2. The model starts correctly but fails to complete the evaluation properly, leading to an infinite sum that is not helpful. The approach lacks clarity and does not arrive at the correct expression in terms of p and q.
      3. The solution is correct and follows the proper steps to evaluate the function at the specified points, leading to the correct final answer.

8-bit:
   Average Score: 8.50/10
   Sample Reasonings:
      1. The solution effectively uses the appropriate formulas and reasoning to arrive at the correct polar coordinates, with a clear explanation of the undefined tangent situation.
      2. 

# XEM CHI TIẾT 1 SAMPLE

In [36]:
# Xem so sánh chi tiết cho sample đầu tiên
print_sample_comparison(
    idx=0,
    dataset_subset=dataset_subset,
    results=results,
    references=references
)


SAMPLE 1 COMPARISON

Problem:
Convert the point $(0,3)$ in rectangular coordinates to polar coordinates.  Enter your answer in the form $(r,\theta),$ where $r > 0$ and $0 \le \theta < 2 \pi.$...

Ground Truth:
We have that $r = \sqrt{0^2 + 3^2} = 3.$  Also, if we draw the line connecting the origin and $(0,3),$ this line makes an angle of $\frac{\pi}{2}$ with the positive $x$-axis.

[asy]
unitsize(0.8 cm);
...

16-bit Answer:
We have the point $(0,3)$ in rectangular coordinates.
To convert to polar coordinates $(r,\theta)$, we use the formulas $x = r \cos \theta$ and $y = r \sin \theta$.
In this case, we have $x=0$ and $y=...

8-bit Answer:
To convert the point $(0,3)$ from rectangular coordinates to polar coordinates, we use the formulas $r = \sqrt{x^2 + y^2}$ and $\theta = \arctan \frac{y}{x}$.
In this case, we have $x = 0$ and $y = 3$...



# So sánh LLM của FP16 với INT4

In [15]:
model_16bit, tokenizer_16bit = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=torch.float16,  # Half precision
    load_in_4bit=False,   # Không quantize
)

# Chuyển sang inference mode (tắt gradient computation để nhanh hơn)
model_16bit = FastLanguageModel.for_inference(model_16bit)

==((====))==  Unsloth 2025.12.8: Fast Gemma3 patching. Transformers: 4.57.1.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.4.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for gemma3 won't work! Using float32.
Unsloth: Gemma3 does not support SDPA - switching to fast eager.
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.96G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/3.64G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/4.69M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/670 [00:00<?, ?B/s]

In [16]:
model_4bit, tokenizer_4bit = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,  # Kích hoạt 4-bit quantization (NF4)
)

# Chuyển sang inference mode
model_4bit=FastLanguageModel.for_inference(model_4bit)

==((====))==  Unsloth 2025.12.8: Fast Gemma3 patching. Transformers: 4.57.1.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.4.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for gemma3 won't work! Using float32.
Unsloth: Gemma3 does not support SDPA - switching to fast eager.


model.safetensors:   0%|          | 0.00/4.56G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/210 [00:00<?, ?B/s]

processor_config.json:   0%|          | 0.00/70.0 [00:00<?, ?B/s]

chat_template.json: 0.00B [00:00, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/4.69M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/670 [00:00<?, ?B/s]

In [23]:
# ================================================================================
# PHẦN 4: TỔNG HỢP KẾT QUẢ
# ================================================================================

def create_results_dict(predictions_16bit, predictions_4bit,
                       references, bleu_metric, rouge_metric):
    """
    Tạo dictionary kết quả cho tất cả models

    Args:
        predictions_16bit: Predictions từ model 16-bit
        predictions_8bit: Predictions từ model 8-bit
        predictions_4bit: Predictions từ model 4-bit
        references: Ground truth references
        bleu_metric: BLEU metric object
        rouge_metric: ROUGE metric object

    Returns:
        dict: Kết quả đầy đủ cho tất cả models
    """
    print("\nĐang tính toán metrics cho tất cả models...\n")

    results = {}

    # Model 16-bit
    print("🔹 Tính metrics cho 16-bit model...")
    metrics_16bit = calculate_metrics(predictions_16bit, references, bleu_metric, rouge_metric)
    results["16-bit"] = {
        "predictions": predictions_16bit,
        "bleu": metrics_16bit["bleu"],
        "rouge_l": metrics_16bit["rouge_l"]
    }
    print(f"   BLEU: {metrics_16bit['bleu']:.2f}, ROUGE-L: {metrics_16bit['rouge_l']:.4f}")

    # Model 4-bit
    print("🔹 Tính metrics cho 4-bit model...")
    metrics_4bit = calculate_metrics(predictions_4bit, references, bleu_metric, rouge_metric)
    results["4-bit"] = {
        "predictions": predictions_4bit,
        "bleu": metrics_4bit["bleu"],
        "rouge_l": metrics_4bit["rouge_l"]
    }
    print(f"   BLEU: {metrics_4bit['bleu']:.2f}, ROUGE-L: {metrics_4bit['rouge_l']:.4f}")

    print("\nHoàn thành tính toán metrics!\n")

    return results


In [24]:
# ================================================================================
# PHẦN 5: LLM-AS-A-JUDGE EVALUATION
# ================================================================================

def evaluate_all_models_with_gpt(problem, predictions_dict, reference, openai_client):
    """
    Sử dụng GPT để đánh giá chất lượng câu trả lời của TẤT CẢ các models cùng lúc

    Args:
        problem: Đề bài toán
        predictions_dict: Dictionary chứa predictions của tất cả models
                         {"16-bit": "answer1", "8-bit": "answer2", "4-bit": "answer3"}
        reference: Lời giải chuẩn
        openai_client: OpenAI client object

    Returns:
        dict: {
            "16-bit": {"score": int, "reasoning": str},
            "8-bit": {"score": int, "reasoning": str},
            "4-bit": {"score": int, "reasoning": str}
        }
    """

    prompt = f"""You are an expert mathematics teacher evaluating student solutions from 3 different AI models.

Problem:
{problem}

Reference Solution (Ground Truth):
{reference}

Model Solutions to Evaluate:

1. Model 16-bit (Base Model):
{predictions_dict.get('16-bit', 'N/A')}

2. Model 4-bit (Quantized):
{predictions_dict.get('4-bit', 'N/A')}

Task: Evaluate EACH model solution on a scale of 1-10, where:
- 1-3: Completely wrong or irrelevant
- 4-5: Partially correct with major errors
- 6-7: Mostly correct with minor errors
- 8-9: Correct with small imperfections
- 10: Perfect solution

Provide your evaluation in JSON format with the following structure:
{{
    "16-bit": {{
        "score": <number 1-10>,
        "reasoning": "<brief explanation in 1-2 sentences>"
    }},
    "4-bit": {{
        "score": <number 1-10>,
        "reasoning": "<brief explanation in 1-2 sentences>"
    }}
}}

Only return valid JSON, nothing else.
"""

    try:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": "You are a mathematics evaluation expert. Always respond with valid JSON only."},
                {"role": "user", "content": prompt}
            ],
            temperature=0.3
        )

        result_text = response.choices[0].message.content.strip()
        result = json.loads(result_text)

        return result

    except Exception as e:
        print(f"Error with GPT API: {e}")
        return {
            "16-bit": {"score": 0, "reasoning": f"API Error: {str(e)}"},
            "4-bit": {"score": 0, "reasoning": f"API Error: {str(e)}"}
        }


def run_llm_judge_evaluation(dataset_subset, results, references, openai_client, max_samples=25):
    """
    Chạy đánh giá LLM-as-a-Judge cho tất cả models

    Args:
        dataset_subset: Dataset đã chuẩn bị
        results: Dictionary kết quả từ create_results_dict()
        references: Ground truth references
        openai_client: OpenAI client object
        max_samples: Số lượng samples tối đa để đánh giá

    Returns:
        dict: Kết quả đánh giá LLM Judge cho tất cả models
    """
    print("\n" + "="*70)
    print(" " * 20 + "LM-AS-A-JUDGE EVALUATION")
    print("="*70)

    llm_judge_results = {
        "16-bit": {"scores": [], "reasonings": []},
        "4-bit": {"scores": [], "reasonings": []}
    }

    num_samples = min(max_samples, len(dataset_subset))
    print(f"\nĐánh giá {num_samples} samples...\n")

    for idx in tqdm(range(num_samples), desc="Evaluating all models"):
        problem = dataset_subset[idx]["problem"]
        reference = references[idx]

        predictions_dict = {
            "16-bit": results["16-bit"]["predictions"][idx],
            "4-bit": results["4-bit"]["predictions"][idx]
        }

        evaluation = evaluate_all_models_with_gpt(problem, predictions_dict, reference, openai_client)

        for model_name in ["16-bit", "4-bit"]:
            if model_name in evaluation:
                llm_judge_results[model_name]["scores"].append(evaluation[model_name]["score"])
                llm_judge_results[model_name]["reasonings"].append(evaluation[model_name]["reasoning"])

    # Tính điểm trung bình
    for model_name in ["16-bit", "4-bit"]:
        scores = llm_judge_results[model_name]["scores"]
        llm_judge_results[model_name]["avg_score"] = np.mean(scores) if scores else 0
        print(f"\n{model_name} - Average LLM Judge Score: {llm_judge_results[model_name]['avg_score']:.2f}/10")

    print("\nHoàn thành đánh giá LLM-as-a-Judge!\n")

    return llm_judge_results


In [26]:
# ================================================================================
# PHẦN 6: HELPER FUNCTIONS
# ================================================================================

def print_sample_comparison(idx, dataset_subset, results, references):
    """
    In ra so sánh chi tiết cho 1 sample

    Args:
        idx: Index của sample
        dataset_subset: Dataset
        results: Kết quả từ create_results_dict()
        references: Ground truth
    """
    print("\n" + "="*70)
    print(f"SAMPLE {idx + 1} COMPARISON")
    print("="*70)

    print(f"\nProblem:\n{dataset_subset[idx]['problem'][:200]}...\n")

    print(f"Ground Truth:\n{references[idx][:200]}...\n")

    print(f"16-bit Answer:\n{results['16-bit']['predictions'][idx][:200]}...\n")

    print(f"4-bit Answer:\n{results['4-bit']['predictions'][idx][:200]}...\n")

    print("="*70)

In [20]:
# Inference với model 16-bit
predictions_16bit = batch_inference(
    model=model_16bit,
    tokenizer=tokenizer_16bit,
    dataset_subset=dataset_subset,
    model_name="16-bit (FP16)"
)

# # Inference với model 8-bit
# predictions_8bit = batch_inference(
#     model=model_8bit,
#     tokenizer=tokenizer_8bit,
#     dataset_subset=dataset_subset,
#     model_name="8-bit (INT8)"
# )

# Inference với model 4-bit
predictions_4bit = batch_inference(
    model=model_4bit,
    tokenizer=tokenizer_4bit,
    dataset_subset=dataset_subset,
    model_name="4-bit (NF4)"
)


Đang inference với 16-bit (FP16)...


Inference 16-bit (FP16): 100%|██████████| 25/25 [14:03<00:00, 33.75s/it]


Hoàn thành inference cho 16-bit (FP16): 25 predictions


Đang inference với 4-bit (NF4)...


Inference 4-bit (NF4): 100%|██████████| 25/25 [20:49<00:00, 49.96s/it]

Hoàn thành inference cho 4-bit (NF4): 25 predictions



In [27]:
import evaluate
bleu_metric = evaluate.load("sacrebleu")
rouge_metric = evaluate.load("rouge")

# Tạo results dictionary với tất cả metrics
results = create_results_dict(
    predictions_16bit=predictions_16bit,
    # predictions_8bit=predictions_8bit,
    predictions_4bit=predictions_4bit,
    references=references,
    bleu_metric=bleu_metric,
    rouge_metric=rouge_metric
)

# In kết quả
print("\n" + "="*70)
print("TỔNG HỢP KẾT QUẢ METRICS")
print("="*70)
for model_name in ["16-bit", "4-bit"]:
    print(f"\n{model_name}:")
    print(f"   BLEU: {results[model_name]['bleu']:.2f}")
    print(f"   ROUGE-L: {results[model_name]['rouge_l']:.4f}")


Đang tính toán metrics cho tất cả models...

🔹 Tính metrics cho 16-bit model...
   BLEU: 19.25, ROUGE-L: 0.2825
🔹 Tính metrics cho 4-bit model...
   BLEU: 18.68, ROUGE-L: 0.2773

Hoàn thành tính toán metrics!


TỔNG HỢP KẾT QUẢ METRICS

16-bit:
   BLEU: 19.25
   ROUGE-L: 0.2825

4-bit:
   BLEU: 18.68
   ROUGE-L: 0.2773


In [28]:
llm_judge_results = run_llm_judge_evaluation(
    dataset_subset=dataset_subset,
    results=results,
    references=references,
    openai_client=client,
    max_samples=10  
)

# In kết quả LLM Judge
print("\n" + "="*70)
print("KẾT QUẢ LLM-AS-A-JUDGE")
print("="*70)
for model_name in ["16-bit", "4-bit"]:
    print(f"\n{model_name}:")
    print(f"   Average Score: {llm_judge_results[model_name]['avg_score']:.2f}/10")
    print(f"   Sample Reasonings:")
    for i, reasoning in enumerate(llm_judge_results[model_name]['reasonings'][:3]):
        print(f"      {i+1}. {reasoning}")



                    LM-AS-A-JUDGE EVALUATION

Đánh giá 10 samples...



Evaluating all models: 100%|██████████| 10/10 [00:30<00:00,  3.03s/it]


16-bit - Average LLM Judge Score: 7.10/10

4-bit - Average LLM Judge Score: 8.90/10

Hoàn thành đánh giá LLM-as-a-Judge!


KẾT QUẢ LLM-AS-A-JUDGE

16-bit:
   Average Score: 7.10/10
   Sample Reasonings:
      1. The solution correctly calculates both the radius and angle for the polar coordinates, providing a clear explanation of the reasoning behind the angle being undefined and concluding with the correct polar coordinates.
      2. The model attempts to manipulate the sums correctly but fails to arrive at a correct final expression for S, leaving the solution incomplete and unclear.
      3. The solution is correct and follows the proper steps to evaluate the function at the specified points, arriving at the correct final answer.

4-bit:
   Average Score: 8.90/10
   Sample Reasonings:
      1. This solution also accurately computes the radius and angle, explaining the undefined tangent and confirming the position on the positive y-axis, leading to the correct polar coordinates.
   

In [ ]:
# Xem so sánh chi tiết cho sample đầu tiên
print_sample_comparison(
    idx=0,
    dataset_subset=dataset_subset,
    results=results,
    references=references
)

# VẼ BIỂU ĐỒ SO SÁNH

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Tạo DataFrame
comparison_df = pd.DataFrame({
    "Model": ["16-bit", "8-bit", "4-bit"],
    "BLEU": [results[m]['bleu'] for m in ["16-bit", "8-bit", "4-bit"]],
    "ROUGE-L": [results[m]['rouge_l'] for m in ["16-bit", "8-bit", "4-bit"]]
})

# Vẽ biểu đồ
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Model Performance Comparison', fontsize=18, fontweight='bold')

# BLEU scores
axes[0].bar(comparison_df["Model"], comparison_df["BLEU"], color=['#2ecc71', '#3498db', '#e74c3c'])
axes[0].set_ylabel('BLEU Score', fontsize=12, fontweight='bold')
axes[0].set_title('BLEU Scores by Model', fontsize=14, fontweight='bold')
for i, v in enumerate(comparison_df["BLEU"]):
    axes[0].text(i, v + 0.5, f'{v:.2f}', ha='center', fontweight='bold')

# ROUGE-L scores
axes[1].bar(comparison_df["Model"], comparison_df["ROUGE-L"], color=['#2ecc71', '#3498db', '#e74c3c'])
axes[1].set_ylabel('ROUGE-L Score', fontsize=12, fontweight='bold')
axes[1].set_title('ROUGE-L Scores by Model', fontsize=14, fontweight='bold')
for i, v in enumerate(comparison_df["ROUGE-L"]):
    axes[1].text(i, v + 0.01, f'{v:.4f}', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('/model_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print("\nĐã lưu biểu đồ: model_comparison.png")

# TÍNH PERFORMANCE DEGRADATION

In [ ]:
print("\n" + "="*70)
print("📊 PERFORMANCE DEGRADATION SO VỚI BASELINE (16-bit)")
print("="*70)

baseline_bleu = results["16-bit"]["bleu"]
baseline_rouge = results["16-bit"]["rouge_l"]

for model_name in ["8-bit", "4-bit"]:
    bleu_degradation = ((results[model_name]["bleu"] - baseline_bleu) / baseline_bleu) * 100
    rouge_degradation = ((results[model_name]["rouge_l"] - baseline_rouge) / baseline_rouge) * 100

    print(f"\n🔹 {model_name}:")
    print(f"   BLEU: {bleu_degradation:+.2f}%")
    print(f"   ROUGE-L: {rouge_degradation:+.2f}%")

print("\n" + "="*70)
print("HOÀN THÀNH BENCHMARK!")
print("="*70)
